In [24]:
# Match_Soldiers_All_By_Children_By_Partitions.py
# 10/20/2025

# Match Soldiers Using children of Jewish Cemeteries Parents

# Rank_Dictionary_Lookup.py
# 10/10/2025
from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd


import json

import sys
import random


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    processing_time_minutes=processing_time_seconds/60
    return f" {processing_time_minutes:.1f}"

def time_now():
    time_now=datetime.datetime.now()  
    return time_now

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def expand_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            new_full_name=re.sub(pattern, rank, full_name.lower(), count=1)
            return new_full_name.title()
    return full_name  # return unchanged if no code found


def strip_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        if pd.isna(full_name):return None
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code              
        if re.search(pattern, full_name):
            new_full_name = re.sub(pattern, "", full_name).strip()   
            return  new_full_name
    return full_name  # return unchanged if no code found


def extract_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found

def extract_rank_multiple(full_name,n):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if len(re.findall(pattern, full_name.lower()))>=n:
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic

blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match Soldiers to Parents By Partitions'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileABMCInput="ABMC_FAG_Soldiers_Merge_Master"
fileSoDInput="star_david_20251019_df"
fileRankInput="Rank_Dictionary"
fileInputCemetery="FAG_Cemeteries_Affiliation_20250928" 

fileABMCIn=DirMain+"0-Master Worksheets\\"+fileABMCInput+".csv"
fileSoDIn=DirMain+"0-Master Worksheets\\"+fileSoDInput+".csv"
fileRankIn=DirMain+"0-Master Worksheets\\"+fileRankInput+".csv"
fileInCemetery=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+fileInputCemetery+".csv"


tic_start_total=get_tic()


soldiers_df=pd.read_csv(fileABMCIn,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
star_david_df=pd.read_csv(fileSoDIn)
rank_df=pd.read_csv(fileRankIn)
cemetery_df=pd.read_csv(fileInCemetery,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• cemetery_df * Download Complete * Size:',len(cemetery_df))

rank_lookup = rank_df.set_index("abbreviated_rank")["rank"].to_dict()
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].str.lower()
# print('Lower:',soldiers_df["soldiers_full_name"])


soldiers_df=soldiers_df.copy()
print("$$$$$ Starting soldiers_full_name Strip Rank $$$") 
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].apply(strip_rank)
# process_time= time_lapse(tic)
# process_time_cum= time_lapse(tic_start)
# tic=get_tic()
print(f"$$$$$ soldiers_full_name Strip Rank Complete $$$ ",'\n')
# print('Strip:',soldiers_df["soldiers_full_name"])
# input('')

# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
# group_A=['Relatives_A0','FAG_Relationship_Relatives_A_','Relatives_A',1,3]

group_A=['Relatives_A0','FAG_Relationship_Names_Graves_A0_','Relatives_A',0,50]
group_B=['Relatives_B','FAG_Relationship_Missing_Name_Graves_POJ_','Relatives_B',1,48]
# group_C=['Relatives_C','FAG_Relationship_Names_Remain_Graves_','Relatives_C',0,99]
group_C=['Relatives_C','FAG_Relationship_Names_Remain_Graves_','Relatives_C',85,99]

# partition_df=pd.DataFrame([group_A,group_B,group_C])
# partition_df=pd.DataFrame([group_A])
# partition_df=pd.DataFrame([group_A1,group_A2,group_A3,group_A4])
# partition_df=pd.DataFrame([group_C,group_A])
partition_df=pd.DataFrame([group_A,group_B,group_C])

partition_df_list=['directory','format','label','range_start','range_end']
partition_df.columns=[partition_df_list]



for index, row in partition_df.iterrows():
    tic_start=get_tic()
    tic=get_tic()
    subdirectory=row['directory']
    file_format=row['format']
    label=row['label']
    partition_start=row['range_start']
    partition_end=row['range_end']
    fileOutRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\Soldier_Parents_Match_"+label+"_"+timestamp+".csv"
    fileMissingCemeteries=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\Missing_Cemeteries_"+label+".csv"
    start=partition_start
    end=partition_end
    parents_soldiers_match_df=pd.DataFrame([])
    tic=get_tic()
    for file_number in range(start,end):
        file_number_str='0'+str(file_number)
        file_number_str=file_number_str[-2:]
        fileInRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv"
        parents_partition_df=pd.read_csv(fileInRelatives,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        parents_partition_df_file_size=len(parents_partition_df)
       
        # %%%%%%%%%%%%%%%%  Drop If No Children's Entry %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        parents_partition_df = parents_partition_df.dropna(subset=['Children'])    
        
        parents_partition_df=drop_sets_dataframe_duplicates(parents_partition_df)
        
        parents_partition_df["children_rank"] = parents_partition_df["Children"].apply(lambda x: extract_rank_multiple(x, 1))
        parents_partition_df=parents_partition_df[parents_partition_df["children_rank"]!='']  
        parents_partition_df.reset_index(drop=True, inplace = True)
               
        if 'plot' not in list(parents_partition_df):parents_partition_df['plot']='No Plot Data'

        print(f" {blank_line}", end='\r')
        print(f"** Group: {label} • Uploading File {file_format+file_number_str} •  Number of Partitions: {end} • Record Count: {len(parents_partition_df):,}/{parents_partition_df_file_size:,}", end='\r') 
        
        print('\n    •• Correlating Cemeteries •• ', end='\r')
        parents_partition_df['cemeteryId'] = pd.to_numeric(parents_partition_df['cemeteryId'], errors='coerce').astype('Int64')       
        cemetery_df_correlation=pd.merge(
        parents_partition_df,
        cemetery_df[['cemetery_id','prob_jewish','low_grave_count','memorial_count']],
        left_on=['cemeteryId'],
        right_on=['cemetery_id'],
        how='left')

        parents_partition_df=cemetery_df_correlation.copy()
        parents_partition_df.drop(columns=['cemetery_id'], inplace=True)

        parents_partition_df["Children"]= parents_partition_df["Children"].astype(str).replace('\.', '', regex=True).str.strip()
    
        parents_short_df_list=['lastName','firstName','middleName','fullName','prob_jewish','cemeteryName','plot','cemeteryId','intermentHasPhoto','birthYear','birthStateName'\
         ,'birthCountryName','Spouses','Children','children_rank','Siblings']        
        soldiers_short_df_list=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','missing_status'\
          ,'birthYear','entry_state','soldiers_cemetery','plot','birthCity','birthState','birthCountry','Parents']
        
        parents_short_df=parents_partition_df[parents_short_df_list]
        soldiers_short_df=soldiers_df[soldiers_short_df_list]       
        matching_parents_soldiers_df=pd.merge(
            soldiers_short_df,
            parents_short_df,
            left_on=['soldiers_last_name','military_rank'],
            right_on=['lastName','children_rank'],    
            how='inner')
        
        matching_parents_soldiers_df_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Parents','lastName','fullName'\
                    ,'Children','children_rank','prob_jewish','cemeteryName','plot_y','cemeteryId','intermentHasPhoto','Spouses','birthStateName'\
                    ,'birthCountryName','missing_status','birthYear_y','soldiers_cemetery'\
                      ,'plot_x','entry_state','birthState','birthCountry','birthYear_x','Siblings']
    
        # $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$   
        print(f" {blank_line}", end='\r')
        print("    •• Starting Needle in Haystack: Soldier - Children: Name  •• ", end='\r')
        needle='soldiers_full_name'
        haystack='Children'
        dataframe_df=matching_parents_soldiers_df[matching_parents_soldiers_df_list]
 
        dataframe_df=dataframe_df.copy()
        dataframe_df['Children']=dataframe_df['Children'].str.lower()
        dataframe_df['soldiers_full_name']=dataframe_df['soldiers_full_name'].str.lower()
        dataframe_df["soldiers_full_name"]=dataframe_df["soldiers_full_name"].apply(strip_rank)
    
        matching_parents_soldiers_short_df, mask_name=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
        matching_parents_soldiers_short_df=matching_parents_soldiers_short_df[matching_parents_soldiers_short_df['Children_match']==True]
        matching_parents_soldiers_short_df.reset_index(drop=True, inplace = True)
    
        process_time= time_lapse(tic)
        process_time_cum= time_lapse(tic_start)
        tic=get_tic()
        print(f" {blank_line}", end='\r')
        print(f"    •• Needle in Haystack: Soldier's Name - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes", end='\r')   
        
        # $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$   
        parents_soldiers_match_partitions_df=pd.merge(
            matching_parents_soldiers_short_df,
            star_david_df,
            left_on=[ 'name_index'],
            right_on=['sod_Name_Index'],
            how='left')
        
        parents_soldiers_match_partitions_df['Star_of_David'] = parents_soldiers_match_partitions_df['sod_Name_Index'].fillna('False')
        parents_soldiers_match_partitions_df['Star_of_David']=parents_soldiers_match_partitions_df['Star_of_David']!='False'
        parents_soldiers_match_partitions_df.drop_duplicates(inplace=True)
        
        reorder_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Children', 'children_rank'\
                      , 'Children_match','prob_jewish','Star_of_David','Parents'\
                      ,'fullName','Spouses','cemeteryName','plot_y','cemeteryId','birthStateName','birthCountryName','birthYear_y'\
                      ,'missing_status','entry_state','birthState','birthCountry','soldiers_cemetery','birthYear_x','plot_x','Siblings']
        
        rename_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Children', 'children_rank'\
                      , 'Children_match','prob_jewish','Star_of_David'\
                     ,'Parents','parent_full_name','spouse','parents_cemetery','plot_parents','cemetery_id'\
                    ,'parent_birth_state','parent_birth_country','parent_birth_year'\
                     ,'missing_status','entry_state','soldier_birth_state','soldier_birth_country','soldiers_cemetery','soldiers_birth_year','soldiers_plot','Siblings']
        
        parents_soldiers_match_partitions_df=parents_soldiers_match_partitions_df[reorder_list]
        parents_soldiers_match_partitions_df.columns=rename_list
        parents_soldiers_match_partitions_df['soldiers_full_name']=parents_soldiers_match_partitions_df['soldiers_full_name'].str.title()
        parents_soldiers_match_partitions_df['Children']=parents_soldiers_match_partitions_df['Children'].str.title()
        parents_soldiers_match_partitions_df['parent_birth_country']=parents_soldiers_match_partitions_df['parent_birth_country'].replace('United States of America', 'USA')
        parents_soldiers_match_partitions_df['soldier_birth_country']=parents_soldiers_match_partitions_df['soldier_birth_country'].replace('United States of America', 'USA')

        parents_soldiers_match_partitions_df['parent_birth_year']=parents_soldiers_match_partitions_df['parent_birth_year'].fillna(0).astype(int)
        parents_soldiers_match_partitions_df['soldiers_birth_year']=parents_soldiers_match_partitions_df['soldiers_birth_year'].fillna(0).astype(int) 
        parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['soldiers_birth_year']-parents_soldiers_match_partitions_df['parent_birth_year']
        parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['age_at_birth'].fillna(0).astype(int)  
        parents_soldiers_match_partitions_df=parents_soldiers_match_partitions_df[parents_soldiers_match_partitions_df['age_at_birth']>=17]
        parents_soldiers_match_partitions_df.reset_index(drop=True, inplace = True)
        
        parents_soldiers_match_df=pd.concat([parents_soldiers_match_df,parents_soldiers_match_partitions_df])
        parents_soldiers_match_df=drop_sets_dataframe_duplicates(parents_soldiers_match_df)
        parents_soldiers_match_df.reset_index(drop=True, inplace = True)
        
        # process_time= time_lapse(tic)
        # process_time_cum= time_lapse(tic_start)
        # tic=get_tic()
        # print(f"** Group: {label} • Uploading File {file_format+file_number_str} Processed •  Number of Partitions: {end} • Record Count: {len(parents_soldiers_match_partitions_df):,}/ {len(parents_soldiers_match_df):,}- Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes") 

        file_parents_soldiers_match_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_soldiers_match_df_"+label+"_"+timestamp+"_Interim.csv"
        parents_soldiers_match_df.to_csv(file_parents_soldiers_match_df,mode="w",index=False,header=True)
        process_time= time_lapse(tic)
        process_time_cum= time_lapse(tic_start)
        tic=get_tic()
        print(f"** Group: {label} • Uploading File {file_format+file_number_str} Processed •  Record Count: {len(parents_soldiers_match_partitions_df):,}/ {len(parents_soldiers_match_df):,} - Process Time: {process_time_cum/60:4.1f} Minutes",'\n') 

          
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
    
    print(' **** Start Saving - parents_soldiers_match_df ')
    file_parents_soldiers_match_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_soldiers_match_df_"+label+"_"+timestamp+".csv"
    parents_soldiers_match_df.to_csv(file_parents_soldiers_match_df,mode="w",index=False,header=True)
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
process_time= time_lapse(tic)
process_time_cum_total= time_lapse(tic_start_total)
tic=get_tic()    
print(f"$$$$$ Parent-Soldiers Classification  - Complete $$$ - Total Processing Time: {process_time_cum_total/60:4.1f} Minutes",'\n' )          


**Starting ***  Match Soldiers to Parents By Partitions  *** , Time: 2025-10-21 18:55:45.013273  **

• cemetery_df * Download Complete * Size: 251399
$$$$$ Starting soldiers_full_name Strip Rank $$$
$$$$$ soldiers_full_name Strip Rank Complete $$$  

** Group: Relatives_A • Uploading File FAG_Relationship_Names_Graves_A0_00 •  Number of Partitions: 50 • Record Count: 2,472/160,899                                                        
** Group: Relatives_A • Uploading File FAG_Relationship_Names_Graves_A0_00 Processed •  Record Count: 1,935/ 1,935 - Process Time:  1.5 Minutes                                            

** Group: Relatives_A • Uploading File FAG_Relationship_Names_Graves_A0_01 •  Number of Partitions: 50 • Record Count: 11,344/895,052                                                       
** Group: Relatives_A • Uploading File FAG_Relationship_Names_Graves_A0_01 Processed •  Record Count: 4,035/ 5,967 - Process Time:  6.5 Minutes                                       

In [ ]:
list(parents_soldiers_match_partitions_df)
parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['soldiers_birth_year']-parents_soldiers_match_partitions_df['parent_birth_year']
parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['age_at_birth'].fillna(0).astype(int)
# print(parents_soldiers_match_partitions_df['age_at_birth'])

parents_soldiers_match_partitions_df=parents_soldiers_match_partitions_df[parents_soldiers_match_partitions_df['age_at_birth']>=17]
parents_soldiers_match_partitions_df['age_at_birth']

In [22]:
        parents_soldiers_match_partitions_df['parent_birth_year']=parents_soldiers_match_partitions_df['parent_birth_year'].fillna(0).astype(int) 
        parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['soldiers_birth_year']-parents_soldiers_match_partitions_df['parent_birth_year']
        parents_soldiers_match_partitions_df['age_at_birth']=parents_soldiers_match_partitions_df['age_at_birth'].fillna(0).astype(int)  
        parents_soldiers_match_partitions_df=parents_soldiers_match_partitions_df[parents_soldiers_match_partitions_df['age_at_birth']>=17]